# Phase 06A.05 — Locked LoRA r16 F1/F3/F8 evaluation
Evaluates one canonical F1-trained adapter under three inference frame policies. F3/F8 are inference-frame transfer arms, not multi-frame-trained adapters.

In [1]:
import gc,json,os,sys,time
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration

In [2]:
RUN_SCOPE='full'; SMOKE_LIMIT=20; FRAME_COUNTS=[1,3,8]
TRAINING_DIR=PROJECT_ROOT/'outputs/phase06a/lora_r16_training'/RUN_SCOPE; ADAPTER_DIR=TRAINING_DIR/'final_adapter'
VALIDATION_CSV=PROJECT_ROOT/'data/splits/phase01/validation.csv'; VALIDATION_IDS=PROJECT_ROOT/'data/splits/phase01/validation_sample_ids.json'
OUTPUT_ROOT=PROJECT_ROOT/'outputs/phase06a/lora_r16_evaluation'
protocol=Phase06AProtocol(run_scope=RUN_SCOPE); limit=SMOKE_LIMIT if RUN_SCOPE=='smoke' else None
assert ADAPTER_DIR.is_dir() and (TRAINING_DIR/'run_manifest.json').is_file()
training_status=json.loads((TRAINING_DIR/'PHASE06A_04_STATUS.json').read_text(encoding='utf-8'))
if RUN_SCOPE=='full': assert training_status['status']=='complete' and training_status['final_training_rows']==EXPECTED_TRAIN_ROWS

## Checkpoint and frozen membership gates

In [3]:
val_df=pd.read_csv(VALIDATION_CSV); frozen_ids=json.loads(VALIDATION_IDS.read_text(encoding='utf-8'))
assert sorted(val_df.sample_id.astype(str))==sorted(map(str,frozen_ids))
assert sha256_file(ADAPTER_DIR/'adapter_model.safetensors')==training_status['adapter_sha256']
if RUN_SCOPE=='full': assert len(val_df)==EXPECTED_VALIDATION_ROWS and sha256_file(VALIDATION_IDS)==EXPECTED_VALIDATION_IDS_SHA256
from peft import PeftModel
base,tokenizer=load_model_and_tokenizer(training=False,attn_implementation=protocol.attention_implementation); base.img_context_token_id=tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
model=PeftModel.from_pretrained(base,ADAPTER_DIR); model.eval()

/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


PeftModel(
  (base_model): LoraModel(
    (model): InternVLChatModel(
      (vision_model): InternVisionModel(
        (embeddings): InternVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        )
        (encoder): InternVisionEncoder(
          (layers): ModuleList(
            (0-23): 24 x InternVisionEncoderLayer(
              (attn): InternAttention(
                (qkv): Linear(in_features=1024, out_features=3072, bias=True)
                (attn_drop): Dropout(p=0.0, inplace=False)
                (proj_drop): Dropout(p=0.0, inplace=False)
                (proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): InternMLP(
                (act): GELUActivation()
                (fc1): Linear(in_features=1024, out_features=4096, bias=True)
                (fc2): Linear(in_features=4096, out_features=1024, bias=True)
              )
              (norm1): LayerNorm((1024,), eps=1e-06,

## Evaluate locked adapter

In [4]:
summaries=[]
for frame_count in FRAME_COUNTS:
    experiment_id=f'L16-F{frame_count}'; out_dir=OUTPUT_ROOT/experiment_id/RUN_SCOPE; out_dir.mkdir(parents=True,exist_ok=True); started=time.perf_counter()
    predictions,metrics=evaluate_fixed_budget_rows(model,tokenizer,val_df,frame_count=frame_count,total_tile_budget=protocol.total_tile_budget,limit=limit)
    expected_ids=val_df.head(limit).sample_id if limit else frozen_ids
    integrity=validate_prediction_artifact(predictions,expected_ids,run_scope=RUN_SCOPE)
    predictions.to_csv(out_dir/'predictions.csv',index=False)
    config={**protocol.to_dict(),'experiment_id':experiment_id,'frame_count':frame_count,'adapter_sha256':training_status['adapter_sha256'],'adapter_training_frames':1,'evaluation_mode':'inference_frame_transfer' if frame_count>1 else 'matched_f1'}
    save_json(out_dir/'config.json',config); save_json(out_dir/'metrics.json',metrics)
    runtime={'elapsed_seconds':time.perf_counter()-started,'peak_vram_gib':torch.cuda.max_memory_allocated()/1024**3}; save_json(out_dir/'runtime.json',runtime)
    write_run_manifest(out_dir/'run_manifest.json',config=config,artifacts=[out_dir/'predictions.csv',out_dir/'metrics.json',out_dir/'runtime.json'])
    summaries.append({'experiment_id':experiment_id,**metrics})
pd.DataFrame([{k:v for k,v in row.items() if k not in {'per_class','confusion_matrix'}} for row in summaries]).to_csv(OUTPUT_ROOT/f'lora_r16_summary_{RUN_SCOPE}.csv',index=False)
save_json(OUTPUT_ROOT/f'PHASE06A_05_STATUS_{RUN_SCOPE}.json',{'phase':'06A.05','status':'complete' if RUN_SCOPE=='full' else 'smoke_complete','rows_per_experiment':len(val_df) if limit is None else limit,'adapter_sha256':training_status['adapter_sha256']})

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


## Cleanup and interpretation constraint

In [5]:
del model,base; gc.collect(); torch.cuda.empty_cache(); display(summaries)

[{'experiment_id': 'L16-F1',
  'rows': 298,
  'accuracy': 0.46308724832214765,
  'macro_f1': 0.43838087493166267,
  'parse_rate': 1.0,
  'per_class': {'A': {'precision': 0.3894736842105263,
    'recall': 0.43023255813953487,
    'f1': 0.4088397790055249,
    'support': 86},
   'B': {'precision': 0.48484848484848486,
    'recall': 0.64,
    'f1': 0.5517241379310345,
    'support': 100},
   'C': {'precision': 0.5208333333333334,
    'recall': 0.35714285714285715,
    'f1': 0.423728813559322,
    'support': 70},
   'D': {'precision': 0.5217391304347826,
    'recall': 0.2857142857142857,
    'f1': 0.36923076923076925,
    'support': 42}},
  'confusion_matrix': {'labels': ['A', 'B', 'C', 'D'],
   'values': [[37, 39, 8, 2],
    [23, 64, 9, 4],
    [22, 18, 25, 5],
    [13, 11, 6, 12]]}},
 {'experiment_id': 'L16-F3',
  'rows': 298,
  'accuracy': 0.47315436241610737,
  'macro_f1': 0.4474939229578111,
  'parse_rate': 1.0,
  'per_class': {'A': {'precision': 0.4148936170212766,
    'recall': 0.45

This notebook does not choose a winner. F3/F8 results cannot be described as multi-frame fine-tuning effects.